# StormEngine V6 — End-to-End Adriatic Forecasting

This notebook launches the reproducible repository pipeline. It uses 239 coastal physical coordinates plus 151 virtual Adriatic sea support points, 2010–2015 training, 2016 validation, 2017 testing, train-only normalization, Natural Earth LSM, station-distance fields, and V6 mean-normalized sea-weighted MSE.

The training implementation lives in `scripts/train.py`, so checkpoints remain compatible between Windows CUDA and Mac CPU/MPS.

In [ ]:
from pathlib import Path
import json, subprocess, sys

here = Path.cwd().resolve()
REPO = here if (here / 'pyproject.toml').exists() else here.parent
assert (REPO / 'pyproject.toml').exists(), 'Open this notebook from StormEngine-DL or its notebooks folder'
print('Repository:', REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{REPO}[notebook]'], check=True)
import yaml

## 1. Set the ERA5 folder on this computer
Change only `ERA5_ROOT`. It must contain the monthly files referenced by `data/manifests/era5_manifest.csv`. On the original layout it is the `DownloadDate` folder beside the repository.

In [ ]:
WINDOWS_ERA5_ROOT = Path(r'D:\Documents\py_projects\StormEngine-DL\DownloadDate')
ERA5_ROOT = WINDOWS_ERA5_ROOT if WINDOWS_ERA5_ROOT.exists() else (REPO.parent / 'DownloadDate').resolve()
BATCH_SIZE = 8  # use 4 or 2 if CUDA reports out-of-memory

config = yaml.safe_load((REPO / 'configs' / 'era5_2010_2017.yaml').read_text(encoding='utf-8'))
config['data']['era5_root'] = str(ERA5_ROOT)
config['training']['batch_size'] = BATCH_SIZE
config['training']['num_workers'] = 0
local_config = REPO / 'configs' / 'era5_2010_2017_windows.local.yaml'
local_config.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('ERA5:', ERA5_ROOT)
print('Local config:', local_config)

## 2. Install and verify CUDA
Install the CUDA-enabled PyTorch build on the Windows computer first if `torch.cuda.is_available()` is false. The command below installs this repository without rebuilding the already-versioned static fields.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Training can be checked on CPU/MPS, but use the Windows CUDA computer for the full run.')

## 3. Data and model preflight
This checks all three chronological splits and one complete 390-point model forward pass before spending time training.

In [ ]:
subprocess.run([sys.executable, '-u', str(REPO / 'scripts' / 'check_data_pipeline.py'), '--config', str(local_config)], cwd=REPO, check=True)

## 4. Small smoke run
Run two training batches and one validation/test batch first. These values only verify the pipeline and are not scientific results. Smoke artifacts are kept separate from the full experiment.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
smoke_command = [
    sys.executable, '-u', str(REPO / 'scripts' / 'train.py'), '--config', str(local_config),
    '--device', DEVICE, '--epochs', '1', '--max-train-batches', '2',
    '--max-eval-batches', '1', '--output-dir', 'artifacts/v6_smoke'
]
subprocess.run(smoke_command, cwd=REPO, check=True)

## 5. Full 2010–2017 training or resume
Set `RESUME=True` after an interrupted run. The best model is selected only by 2016 validation loss; 2017 test data are evaluated once after training.

In [ ]:
RESUME = False
command = [sys.executable, '-u', str(REPO / 'scripts' / 'train.py'), '--config', str(local_config), '--device', DEVICE]
last_checkpoint = REPO / 'artifacts' / 'v6_2010_2017' / 'last.pt'
if RESUME:
    assert last_checkpoint.exists(), last_checkpoint
    command += ['--resume', str(last_checkpoint)]
print('Running:', ' '.join(map(str, command)))
subprocess.run(command, cwd=REPO, check=True)

## 6. Inspect curves and final test metrics

In [ ]:
import matplotlib.pyplot as plt
artifact_dir = REPO / 'artifacts' / 'v6_2010_2017'
history = json.loads((artifact_dir / 'history.json').read_text(encoding='utf-8'))
metrics = json.loads((artifact_dir / 'metrics.json').read_text(encoding='utf-8'))
epochs = [row['epoch'] for row in history]
plt.figure(figsize=(8, 4))
plt.plot(epochs, [row['train_loss'] for row in history], label='train')
plt.plot(epochs, [row['validation_loss'] for row in history], label='validation')
plt.xlabel('Epoch'); plt.ylabel('V6 weighted MSE (normalized)'); plt.grid(alpha=.3); plt.legend(); plt.show()
print(json.dumps(metrics, indent=2))